In [0]:
%pip install aiohttp beautifulsoup4 lxml pandas
dbutils.library.restartPython()

In [0]:
import pandas as pd

In [0]:
BASE_URL = "https://books.toscrape.com/"

In [0]:
# Importamos la biblioteca necesaria para hacer peticiones HTTP asíncronas
import aiohttp

# Definimos una función asíncrona indicando la URL a la que queremos acceder
async def fetch_html(url):
    
    # Abrimos una sesión HTTP. 'async with' garantiza que los recursos 
    # se liberen y la sesión se cierre automáticamente al terminar.
    async with aiohttp.ClientSession() as session:
        
        # Realizamos la petición HTTP tipo GET a la URL indicada
        async with session.get(url) as response:
            
            # Verificamos que la respuesta sea exitosa (código 200). 
            # Si hay un error (ej. 404 No encontrado), esto lanza una excepción.
            response.raise_for_status()
            
            # Esperamos asíncronamente a que se descargue el cuerpo de la página 
            # y lo devolvemos en formato de texto (string).
            return await response.text()

In [0]:
html_content = await fetch_html(BASE_URL)
print(html_content)

In [0]:
# Importamos la clase BeautifulSoup desde la biblioteca bs4
from bs4 import BeautifulSoup

# Creamos el objeto 'soup' (sopa). Le pasamos dos cosas:
# 1. 'html_content': la variable que contiene todo el texto HTML (lo que descargamos antes).
# 2. '"lxml"': es el motor que lee y estructura el HTML. Se usa porque es muy rápido y eficiente.
soup = BeautifulSoup(html_content, "lxml")

# Navegamos por la estructura del HTML hasta la etiqueta <title>.
# Al agregar '.text', le decimos que ignore las etiquetas HTML (<title>...</title>) 
# y nos devuelva únicamente el texto limpio que está adentro. Finalmente, lo imprimimos.
print(soup.title.text)

In [0]:
# Creamos una lista vacía donde guardaremos la información de cada libro.
books = []

# 'soup.select' usa selectores CSS.
# Aquí busca TODAS las etiquetas <article> que tengan la clase "product_pod".
# El bucle 'for' va a recorrer cada uno de esos artículos (cada artículo es un libro).
for article in soup.select("article.product_pod"):
    
    # Dentro de ese artículo específico, busca la etiqueta <h3>, luego la etiqueta <a> (enlace),
    # y extrae el valor del atributo "title" (que contiene el nombre completo del libro).
    title = article.h3.a["title"]
    
    # 'select_one' busca solo el PRIMER elemento que coincida con el selector CSS.
    # Aquí busca algo con la clase "price_color", extrae su texto y usa '.strip()' 
    # para limpiar cualquier espacio en blanco o salto de línea al principio y al final.
    price_text = article.select_one(".price_color").text.strip()
    
    # Hace exactamente lo mismo que el precio, pero buscando la clase "availability".
    availability = article.select_one(".availability").text.strip()
    
    # Crea un diccionario con los datos limpios de este libro en particular 
    # y lo añade (append) a nuestra lista 'books'.
    books.append({
        "product_name": title,
        "price_text": price_text,
        "availability": availability,
        "source": "books_toscrape" # Añade un dato fijo para saber de dónde salió la info
    })

# Imprime en consola cuántos diccionarios (libros) se guardaron en la lista en total.
print(f"Libros encontrados: {len(books)}")

# Muestra los elementos de la lista
books

In [0]:
df = pd.DataFrame(books)
    
df["price"] = df["price_text"].str.extract(r'([\d.]+)').astype(float)
df["currency"] = "GBP"
df["in_stock"] = df["availability"].str.contains("In stock", case=False)

df = df[["product_name", "price", "currency", "in_stock", "source"]]

print("¡Éxito! Los datos han sido estructurados.")
df

In [0]:
import asyncio
import aiohttp
import pandas as pd
import re
from bs4 import BeautifulSoup

# --- 1. FASE DE DESCARGA (ASÍNCRONA) ---
async def fetch_page(session, url, semaphore):
    """Descarga el HTML de una página de forma segura, respetando el límite de peticiones."""
    async with semaphore:
        try:
            async with session.get(url) as response:
                response.raise_for_status()
                return await response.text()
        except Exception as e:
            print(f"Error al descargar {url}: {e}")
            return None

# --- 2. FASE DE EXTRACCIÓN (SÍNCRONA) ---
def parse_books(html):
    """Recibe un HTML, usa BeautifulSoup y devuelve una lista de diccionarios con los libros."""
    if not html:
        return []
    
    soup = BeautifulSoup(html, "lxml")
    books_on_page = []
    
    for article in soup.select("article.product_pod"):
        title = article.h3.a["title"]
        price_text = article.select_one(".price_color").text.strip()
        availability = article.select_one(".availability").text.strip()
        
        books_on_page.append({
            "product_name": title,
            "price_text": price_text,
            "availability": availability,
            "source": "books_toscrape"
        })
        
    return books_on_page

# --- 3. ORQUESTADOR PRINCIPAL ---
async def main():
    print("Iniciando la extracción de 50 páginas...")
    
    base_url = "http://books.toscrape.com/catalogue/page-{}.html"
    urls = [base_url.format(i) for i in range(1, 51)]
    
    semaphore = asyncio.Semaphore(5) # Importante: limitamos el número de peticiones simultáneas
    
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_page(session, url, semaphore) for url in urls]
        html_pages = await asyncio.gather(*tasks)
    
    print("Descarga completada. Procesando los datos...")
    
    all_books = []
    for html in html_pages:
        all_books.extend(parse_books(html))
        
    print(f"Total de libros extraídos: {len(all_books)}")

    # --- 4. FASE DE LIMPIEZA ---
    df = pd.DataFrame(all_books)
    
    df["price"] = df["price_text"].str.extract(r'([\d.]+)').astype(float)
    df["currency"] = "GBP"
    df["in_stock"] = df["availability"].str.contains("In stock", case=False)
    
    df = df[["product_name", "price", "currency", "in_stock", "source"]]
    
    print("¡Éxito! Los datos han sido estructurados.")
    
    # NUEVO: Retornamos el DataFrame para que no se pierda al terminar la función
    return df

# --- 5. EJECUCIÓN EN DATABRICKS ---
# Ejecutamos la función y guardamos el resultado en una variable global
df_books = await main()

In [0]:
books_spark_df = spark.createDataFrame(df_books)

books_spark_df.write.mode("overwrite").format("delta").saveAsTable("bronze.web_books")

In [0]:
%sql
-- select * from bronze.web_books
select count(1) from bronze.web_books